In [1]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from env.charging_env import ChargingStationEnv
from metrics.validate import report_queueing_laws, validate_episode
from offline_opt import (
    StationSpec,
    build_offline_model,
    build_relaxed_model,
    default_horizon_minutes,
    extract_solution,
    solve_offline_model,
    taper_time_constant,
    vehicles_from_evs,
)
from policy.power.proportional import ProportionalPower
from policy.power.static import StaticPower
from policy.queue.fifo import FIFOQueuePolicy
from policy.queue.lowest_soc_diff import LowestSoCDiffQueuePolicy
from visualization.ev_curves import plot_ev_theory_vs_sim
from visualization.pile_power import plot_pile_dispenser_power


In [ ]:
# Shared station / traffic inputs (simulation and offline use the same layout).
SCENARIO = {
    "n_piles": 1,
    "n_dispensers": 2,
    "n_modules": 6,
    "p_module": 25.0,
    "queue_capacity": 100,
    "mean_interarrival": 20.0,
}

# Simulation controls
SEED = 42
POLICY_SEED = 1
QUEUE_POLICY = FIFOQueuePolicy()
POWER_POLICY = ProportionalPower()



In [3]:
env = ChargingStationEnv(
    **SCENARIO,
    power_policy=POWER_POLICY,
)

obs, info = env.reset(seed=SEED)
done = False
rng = np.random.default_rng(POLICY_SEED)
policy = QUEUE_POLICY
total_reward = 0.0

t0 = time.perf_counter()
while not done:
    mask = env.action_masks()
    ev, pile_id = policy.decide(obs, mask, rng, env.engine.station)
    obs, reward, done, _, info = env.step(pile_id, ev=ev)
    total_reward += reward
sim_runtime = time.perf_counter() - t0

finished = env.engine.metrics.finished_evs
arrived = env.engine.metrics.arrived_evs
dropped = env.engine.metrics.dropped_evs
sim_total_sojourn = float(
    sum(ev.departure_time - ev.arrival_time for ev in finished)
)
sim_mean_sojourn = env.engine.metrics.mean_sojourn()

In [4]:
print(f"Finished EVs: {len(finished)}")
print(f"Arrived EVs:  {len(arrived)}")
print(f"Dropped EVs:  {len(dropped)}")
print(f"Avg L: {env.engine.metrics.average_L():.3f}")
print(f"Avg Q: {env.engine.metrics.average_Q():.3f}")
print(f"Total reward: {total_reward:.3f}")
print(f"Sim time (episode clock): {env.engine.current_time:.1f} min")
print(f"Sim wall-clock runtime:   {sim_runtime:.3f} s")
print(f"Sim total sojourn (finished): {sim_total_sojourn:.2f} min")
print(f"Sim mean sojourn (finished):  {sim_mean_sojourn:.2f} min")

Finished EVs: 65
Arrived EVs:  76
Dropped EVs:  0
Avg L: 9.686
Avg Q: 7.880
Total reward: -11346.544
Sim time (episode clock): 1440.0 min
Sim wall-clock runtime:   0.092 s
Sim total sojourn (finished): 12722.72 min
Sim mean sojourn (finished):  195.73 min


In [9]:
# Offline IP + continuous relaxations on the same station layout and the same
# arrival stream the episode already realized (no synthetic arrivals after
# simulation end). VehicleData uses each EV's arrival, battery, s_i, s_f.
offline_station = StationSpec(
    n_piles=SCENARIO["n_piles"],
    n_dispensers=SCENARIO["n_dispensers"],
    n_modules=SCENARIO["n_modules"],
    p_module=SCENARIO["p_module"],
)

# Offline MILP / RP controls (same arrivals + station as the finished episode)
DELTA = 5.0
MIP_GAP = 1e-3
TIME_LIMIT = 600.0  # seconds per solve; raise if the gap does not close
# Manual completion horizon for the offline models (minutes). Must be long
# enough that every arrived EV can finish. Arrival list is cut at sim end;
# this horizon is only the MILP's planning window T, not an arrival generator.
OFFLINE_HORIZON = 2000.0

sim_end = float(env.engine.current_time)
# Arrivals that occurred by simulation end only (engine already stops at
# MAX_TIME / SIM_OVER; the filter makes the cutoff explicit).
offline_evs = [ev for ev in arrived if ev.arrival_time <= sim_end + 1e-9]
vehicles = vehicles_from_evs(offline_evs)
tau = taper_time_constant()

# Manual planning horizon T for the MILP (must cover finishing every vehicle).
horizon = float(OFFLINE_HORIZON)
suggested = default_horizon_minutes(vehicles, offline_station)
if horizon < suggested:
    print(
        f"Warning: OFFLINE_HORIZON={horizon:g} < suggested "
        f"default_horizon_minutes={suggested:.1f}; the model may be infeasible."
    )

full_cap = offline_station.n_modules * offline_station.p_module
reduced_cap = (
    offline_station.n_modules - offline_station.n_dispensers + 1
) * offline_station.p_module

print(
    f"Offline instance: J={len(vehicles)} vehicles "
    f"(arrived by sim_end={sim_end:.1f} min), "
    f"horizon T={horizon:.1f} min, delta={DELTA:g} min, "
    f"full_cap={full_cap:g} kW, reduced_cap={reduced_cap:g} kW"
)

Offline instance: J=76 vehicles (arrived by sim_end=1440.0 min), horizon T=2000.0 min, delta=5 min, full_cap=150 kW, reduced_cap=125 kW


In [ ]:
ip_model = build_offline_model(
    vehicles,
    offline_station,
    delta=DELTA,
    horizon_minutes=horizon,
    tau=tau,
    tie_break=False,
)
solve_offline_model(ip_model, mip_gap=MIP_GAP, time_limit=TIME_LIMIT)
ip_sol = extract_solution(ip_model)
print("Done solving IP")

In [10]:
rp_lo_model = build_relaxed_model(
    vehicles,
    offline_station,
    delta=DELTA,
    horizon_minutes=horizon,
    tau=tau,
    pile_capacity_kw=full_cap,
)
solve_offline_model(rp_lo_model, mip_gap=MIP_GAP, time_limit=TIME_LIMIT)
rp_lower = extract_solution(rp_lo_model)
print("Done solving RP_lower.")

RuntimeError: Offline MILP produced no feasible solution (status=9). If this was a time limit, raise time_limit or relax mip_gap.

In [ ]:
rp_up_model = build_relaxed_model(
    vehicles,
    offline_station,
    delta=DELTA,
    horizon_minutes=horizon,
    tau=tau,
    pile_capacity_kw=reduced_cap,
)
solve_offline_model(rp_up_model, mip_gap=MIP_GAP, time_limit=TIME_LIMIT)
rp_upper = extract_solution(rp_up_model)
print("Done solving RP_upper.")


In [ ]:
# Solver status and MIP gap for the offline models
status_gap = pd.DataFrame(
    [
        {
            "model": "IP",
            "status": ip_sol.status,
            "mip_gap": ip_sol.mip_gap,
        },
        {
            "model": "RP_lower (full cap)",
            "status": rp_lower.status,
            "mip_gap": rp_lower.mip_gap,
        },
        {
            "model": "RP_upper (reduced cap)",
            "status": rp_upper.status,
            "mip_gap": rp_upper.mip_gap,
        },
    ]
).set_index("model")
status_gap


In [ ]:
# Runtime and sojourn: offline models vs the causal simulation.
# Offline totals include every EV that arrived by sim end (MILP forces finish
# by OFFLINE_HORIZON). Simulation totals are over finished EVs only; EVs still
# plugged or waiting at sim end are excluded from sim sojourn.
compare = pd.DataFrame(
    [
        {
            "model": "RP_lower",
            "status": rp_lower.status,
            "runtime_s": rp_lower.runtime,
            "total_sojourn_min": rp_lower.total_sojourn,
            "mean_sojourn_min": rp_lower.mean_sojourn,
            "n_vehicles": rp_lower.n_vehicles,
        },
        {
            "model": "IP",
            "status": ip_sol.status,
            "runtime_s": ip_sol.runtime,
            "total_sojourn_min": ip_sol.total_sojourn,
            "mean_sojourn_min": ip_sol.mean_sojourn,
            "n_vehicles": ip_sol.n_vehicles,
        },
        {
            "model": "RP_upper",
            "status": rp_upper.status,
            "runtime_s": rp_upper.runtime,
            "total_sojourn_min": rp_upper.total_sojourn,
            "mean_sojourn_min": rp_upper.mean_sojourn,
            "n_vehicles": rp_upper.n_vehicles,
        },
        {
            "model": "Simulation",
            "status": "-",
            "runtime_s": sim_runtime,
            "total_sojourn_min": sim_total_sojourn,
            "mean_sojourn_min": sim_mean_sojourn,
            "n_vehicles": len(finished),
        },
    ]
).set_index("model")

print(
    f"Arrived by sim end={len(offline_evs)}, finished={len(finished)}, "
    f"dropped={len(dropped)}, still in system at end="
    f"{len(offline_evs) - len(finished) - len(dropped)}."
)
print(
    "Expected sandwich when IP is optimal:\n"
    "  RP_lower.total_sojourn <= IP <= RP_upper, and IP <= Simulation "
    "(on the same finished set / when the episode clears everyone)."
)
compare


In [ ]:
summary = env.engine.metrics.queueing_summary(env.engine.current_time)
waits, services, sojourns = env.engine.metrics.finished_time_arrays()
rep_val = validate_episode(env, verbose=True)
reo_q = report_queueing_laws(env, verbose=True)


In [ ]:
fig = plot_pile_dispenser_power(
    env,
    0,
    t_start=450,
    t_end=600,
    show_theory=True,
    show_sim_bms=True,
    show_actual=True,
    show_legend=True,
    label_theory="Unconstrained",
    label_sim_bms="BMS Request",
    label_actual="Actual",
    label_charge_steps="Allotted Power",
    show_charge_change_lines=True,
    show_charge_change_vlines=True,
    show_trace_labels=False,
    title=None,
    show_event_lines=True,
    event_line_alpha=0.25,
    show_ev_ids=True,
)
fig


In [ ]:
figs = plot_ev_theory_vs_sim(
    env,
    ev_ids=[10 + i for i in range(5)],
    charts=["P-S", "T-S", "P-T"],
    show_theory=True,
    show_sim_bms=True,
    show_sim_actual=True,
    show_recorded_samples=False,
    label_theory=None,
    label_sim_bms=None,
    label_sim_actual=None,
    label_recorded=None,
)
figs
